# 20 · End-to-End Pipelines

Run all 26 built-in domain pipelines from search to output.

## Contents
1. All 26 pipelines overview
2. Running pipelines
3. Domain pipeline catalogue
4. Custom pipelines
5. Scheduling

## 1 · All 26 Pipelines

In [ ]:
from pygeovision.ai.pipelines.domains import list_pipelines, _PIPELINE_REGISTRY
pipes = list_pipelines()
print(f'PyGeoVision End-to-End Pipelines ({len(pipes)}):')
print()
print(f'  {"Pipeline":<35} {"Domain":<15} Description')
print('  ' + '-' * 80)
for name, cls in _PIPELINE_REGISTRY.items():
    if cls is not None:
        obj = cls(None)
        print(f'  {name:<35} {obj.domain:<15} {obj.description[:45]}')

PyGeoVision End-to-End Pipelines (26):

  Pipeline                            Domain          Description
  ────────────────────────────────────────────────────────────────────────────────
  crop_type_mapping                   agriculture     Sentinel-2 time series -> crop type
  crop_health                         agriculture     NDVI time-series -> crop health
  irrigation_detection                agriculture     Multispectral -> irrigation mapping
  canopy_height                       forestry        Sentinel-2 -> canopy height
  tree_species                        forestry        Hyperspectral -> tree species
  forest_fire                         wildfire        Thermal -> active fire mapping
  road_extraction                     urban           High-res aerial -> road network
  infrastructure_monitoring           urban           Bi-temporal -> infrastructure change
  flood_mapping                       disaster        SAR/optical -> flood extent
  water_quality                    

## 2 · Running Domain Pipelines

In [ ]:
import pygeovision as pgv
client = pgv.PyGeoVision()

NYC = (-74.1, 40.6, -73.7, 40.9)

# Single-date pipeline
result = client.pipeline(
    'building_footprints',
    bbox=NYC,
    output_dir='./results/pipelines/buildings/',
    date='2024-06',
)
status = 'Success' if result.success else 'Failed'
print(f'Building Footprints: {status}')
if result.success:
    print(f'  Output:   {result.output_path}')
    print(f'  Duration: {result.duration_seconds:.1f}s')
else:
    print(f'  Error: {result.error}')
print()

# Bi-temporal pipeline
result2 = client.pipeline(
    'change_detection',
    bbox=NYC,
    output_dir='./results/pipelines/change/',
    date_before='2018-01',
    date_after='2024-01',
)
print(f'Change Detection:    {"Success" if result2.success else "Failed"}')
if result2.success:
    print(f'  Period: 2018 -> 2024')
    print(f'  Output: {result2.output_path}')

## 3 · Domain Catalogue

In [ ]:
domains = {
    'Agriculture':  ['crop_type_mapping','crop_health','irrigation_detection','vegetation_indices','crop_monitoring'],
    'Forestry':     ['canopy_height','tree_species','forest_fire','deforestation'],
    'Urban':        ['building_footprints','road_extraction','infrastructure_monitoring','urban_growth'],
    'Water/Coast':  ['water_bodies','flood_mapping','water_quality','coastal_monitoring','ocean_ship_detection'],
    'Disaster':     ['disaster_assessment','landslide_detection','volcano_monitoring'],
    'Climate':      ['land_cover','land_surface_temperature','carbon_estimation'],
}
print('Pipeline catalogue by domain:')
for domain, pipes in domains.items():
    print(f'  {domain}:')
    for p in pipes:
        print(f'    client.pipeline("{p}", bbox=..., date=...)')

## 4 · Pipeline Results

In [ ]:
from pygeovision.ai.pipelines.domains import PipelineResult
from pathlib import Path

# Inspect a PipelineResult
r = PipelineResult(
    name='building_footprints',
    success=True,
    output_path=Path('./results/buildings.geojson'),
    stats={'n_buildings': 3947, 'total_area_ha': 48.3, 'mean_area_m2': 367},
    duration_seconds=145.2,
)
print('PipelineResult fields:')
print(f'  .name:             {r.name}')
print(f'  .success:          {r.success}')
print(f'  .output_path:      {r.output_path}')
print(f'  .stats:            {r.stats}')
print(f'  .duration_seconds: {r.duration_seconds}')
print(f'  str(r):            {r}')

## 5 · Scheduling Production Pipelines

In [ ]:
MONITORING = {
    '0 6 * * 1':    'building_footprints',
    '0 0 1 * *':    'deforestation',
    '0 */6 * * *':  'flood_mapping',
    '0 8 * * *':    'crop_health',
    '0 12 * * 3':   'water_quality',
    '0 0 * * 0':    'ocean_ship_detection',
}
print('Production Pipeline Schedule:')
print(f'  {"Cron":<22} Pipeline')
print('  ' + '-' * 45)
for cron, pipeline in MONITORING.items():
    print(f'  {cron:<22} {pipeline}')
print()
print('Schedule via PyGeoFetch:')
print("  client.data.schedule_pipeline('./pipeline.yaml', cron='0 6 * * 1')")